In [1]:
import pandas as pd 
import matplotlib.pyplot as plt
df_2022 = pd.read_csv("/home/priyansh_jain01/workspace/Air_quality_project/data/raw/2022.csv")
df_2023 = pd.read_csv("/home/priyansh_jain01/workspace/Air_quality_project/data/raw/2023.csv")
df_2024 = pd.read_csv("/home/priyansh_jain01/workspace/Air_quality_project/data/raw/2024.csv")
df_2025 = pd.read_csv("/home/priyansh_jain01/workspace/Air_quality_project/data/raw/2025.csv")

combined_df = pd.concat(
    [df_2022, df_2023, df_2024, df_2025],
    ignore_index=True
)


combined_df["Timestamp"] = pd.to_datetime(
    combined_df["Timestamp"],
    format="mixed"
)


combined_df["Year"] = combined_df["Timestamp"].dt.year
combined_df["Month"] = combined_df["Timestamp"].dt.month
combined_df["Month_Name"] = combined_df["Timestamp"].dt.month_name()


cleaned_df = combined_df.copy()

In [2]:
cleaned_df.shape

(1461, 28)

# Step 1 of cleaning: Drop completely empty columns

In [3]:
cleaned_df = cleaned_df.dropna(axis=1, how="all")

In [4]:
cleaned_df.shape

(1461, 22)

In [5]:
cleaned_df.columns.tolist()

['Timestamp',
 'PM2.5 (µg/m³)',
 'PM10 (µg/m³)',
 'NO (µg/m³)',
 'NO2 (µg/m³)',
 'NOx (ppb)',
 'NH3 (µg/m³)',
 'SO2 (µg/m³)',
 'CO (mg/m³)',
 'Ozone (µg/m³)',
 'Benzene (µg/m³)',
 'Toluene (µg/m³)',
 'AT (°C)',
 'RH (%)',
 'WS (m/s)',
 'WD (deg)',
 'TOT-RF (mm)',
 'SR (W/mt2)',
 'BP (mmHg)',
 'Year',
 'Month',
 'Month_Name']

In [ ]:
cleaned_df.nunique().sort_values()  

TOT-RF (mm)           1
Year                  4
Month                12
Month_Name           12
WS (m/s)            515
CO (mg/m³)          682
Benzene (µg/m³)     726
BP (mmHg)           886
Toluene (µg/m³)    1145
AT (°C)            1207
SO2 (µg/m³)        1251
Ozone (µg/m³)      1308
RH (%)             1317
NH3 (µg/m³)        1330
NO2 (µg/m³)        1365
NO (µg/m³)         1377
SR (W/mt2)         1384
NOx (ppb)          1387
PM2.5 (µg/m³)      1394
WD (deg)           1402
PM10 (µg/m³)       1410
Timestamp          1461
dtype: int64

# Cleaning Step 2 — Remove the constant column

In [7]:
cleaned_df = cleaned_df.drop(columns=["TOT-RF (mm)"])

In [8]:
cleaned_df.shape

(1461, 21)

# Cleaning Step 3 — Check duplicates

In [9]:
cleaned_df.duplicated().sum()

np.int64(0)

In [10]:
cleaned_df["Timestamp"].duplicated().sum()

np.int64(0)

In [11]:
"TOT-RF (mm)" in cleaned_df.columns

False

In [12]:
cleaned_df.shape

(1461, 21)

In [13]:
cleaned_df.isnull().sum().sort_values(ascending=True)

Timestamp            0
Month_Name           0
Year                 0
Month                0
SR (W/mt2)          30
RH (%)              33
AT (°C)             33
PM2.5 (µg/m³)       34
WD (deg)            34
PM10 (µg/m³)        38
Ozone (µg/m³)       39
NOx (ppb)           41
NH3 (µg/m³)         41
WS (m/s)            42
NO (µg/m³)          46
NO2 (µg/m³)         46
SO2 (µg/m³)         51
CO (mg/m³)          70
BP (mmHg)          167
Benzene (µg/m³)    226
Toluene (µg/m³)    228
dtype: int64

# Convert percentages into actual counts

In [18]:
missing_summary = pd.DataFrame({
    "Missing_Count": cleaned_df.isnull().sum(),
    "Missing_Percentage": cleaned_df.isnull().mean() * 100
})

missing_summary = missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values("Missing_Percentage", ascending=False)

missing_summary

,Missing_Count,Missing_Percentage
Toluene (µg/m³),228,15.605749
Benzene (µg/m³),226,15.468857
BP (mmHg),167,11.430527
CO (mg/m³),70,4.791239
SO2 (µg/m³),51,3.490760
NO2 (µg/m³),46,3.148528
NO (µg/m³),46,3.148528
WS (m/s),42,2.874743
NOx (ppb),41,2.806297
NH3 (µg/m³),41,2.806297


In [33]:
cleaned_df[
    cleaned_df["PM2.5 (µg/m³)"].isnull()
][["Timestamp", "Year", "Month", "Month_Name", "PM2.5 (µg/m³)", "PM10 (µg/m³)"]]



,Timestamp,Year,Month,Month_Name,PM2.5 (µg/m³),PM10 (µg/m³)
559,2023-07-14,2023,7,July,NaN,NaN
560,2023-07-15,2023,7,July,NaN,NaN
561,2023-07-16,2023,7,July,NaN,NaN
562,2023-07-17,2023,7,July,NaN,NaN
563,2023-07-18,2023,7,July,NaN,NaN
564,2023-07-19,2023,7,July,NaN,NaN
584,2023-08-08,2023,8,August,NaN,NaN
585,2023-08-09,2023,8,August,NaN,NaN
586,2023-08-10,2023,8,August,NaN,NaN
587,2023-08-11,2023,8,August,NaN,NaN


# Count missing PM2.5,PM10,NO2,SO2,CO,Ozone by year

In [22]:
cleaned_df.groupby("Year")["PM2.5 (µg/m³)"].apply(
    lambda x: x.isnull().sum()
)

Year
2022     0
2023    10
2024    22
2025     2
Name: PM2.5 (µg/m³), dtype: int64

In [23]:
cleaned_df.groupby("Year")["PM10 (µg/m³)"].apply(
    lambda x: x.isnull().sum()
)

Year
2022     0
2023    13
2024    24
2025     1
Name: PM10 (µg/m³), dtype: int64

In [26]:
cleaned_df.groupby("Year")["NO2 (µg/m³)"].apply(
    lambda x: x.isnull().sum()
)

Year
2022     4
2023    19
2024    23
2025     0
Name: NO2 (µg/m³), dtype: int64

In [51]:
cleaned_df.groupby("Year")["SO2 (µg/m³)"].apply(
    lambda x: x.isnull().sum()
)

Year
2022     1
2023    21
2024    24
2025     5
Name: SO2 (µg/m³), dtype: int64

In [ ]:
cleaned_df.groupby("Year")["CO (mg/m³)"].apply(
    lambda x: x.isnull().sum()
)

Year
2022    28
2023    18
2024    24
2025     0
Name: CO (mg/m³), dtype: int64

In [31]:
cleaned_df.groupby("Year")["Ozone (µg/m³)"].apply(
    lambda x: x.isnull().sum()
)

Year
2022     0
2023    11
2024    27
2025     1
Name: Ozone (µg/m³), dtype: int64

# CORE POLLUTION VARIABLES
PM2.5,
PM10,
NO2,
SO2,
CO,
Ozone,
       
(Treat carefully)


# WEATHER VARIABLES
AT,
RH,
WS,
WD,
SR,
BP,
       
(Potentially useful for relationships/ML)


# LESS CENTRAL VARIABLES
Benzene,
Toluene,
       
(~15% missing
Need to decide whether they're necessary for our project)

# Display the dates where PM2.5 is missing

In [35]:
cleaned_df.loc[
    cleaned_df["PM2.5 (µg/m³)"].isnull(),
    ["Timestamp", "Year", "Month", "Month_Name", "PM2.5 (µg/m³)"]
]

,Timestamp,Year,Month,Month_Name,PM2.5 (µg/m³)
559,2023-07-14,2023,7,July,NaN
560,2023-07-15,2023,7,July,NaN
561,2023-07-16,2023,7,July,NaN
562,2023-07-17,2023,7,July,NaN
563,2023-07-18,2023,7,July,NaN
564,2023-07-19,2023,7,July,NaN
584,2023-08-08,2023,8,August,NaN
585,2023-08-09,2023,8,August,NaN
586,2023-08-10,2023,8,August,NaN
587,2023-08-11,2023,8,August,NaN


# Display the dates where PM10 is missing

In [36]:
cleaned_df.loc[
    cleaned_df["PM10 (µg/m³)"].isnull(),
    ["Timestamp", "Year", "Month", "PM10 (µg/m³)"]
]

,Timestamp,Year,Month,PM10 (µg/m³)
521,2023-06-06,2023,6,NaN
551,2023-07-06,2023,7,NaN
559,2023-07-14,2023,7,NaN
560,2023-07-15,2023,7,NaN
561,2023-07-16,2023,7,NaN
562,2023-07-17,2023,7,NaN
563,2023-07-18,2023,7,NaN
564,2023-07-19,2023,7,NaN
584,2023-08-08,2023,8,NaN
585,2023-08-09,2023,8,NaN


# Check whether PM2.5 and PM10 are missing together

In [39]:
cleaned_df[
    cleaned_df["PM2.5 (µg/m³)"].isnull()
    & cleaned_df["PM10 (µg/m³)"].isnull()
][["Timestamp", "PM2.5 (µg/m³)", "PM10 (µg/m³)"]]



,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³)
559,2023-07-14,NaN,NaN
560,2023-07-15,NaN,NaN
561,2023-07-16,NaN,NaN
562,2023-07-17,NaN,NaN
563,2023-07-18,NaN,NaN
564,2023-07-19,NaN,NaN
584,2023-08-08,NaN,NaN
585,2023-08-09,NaN,NaN
586,2023-08-10,NaN,NaN
587,2023-08-11,NaN,NaN


In [40]:
(
    cleaned_df["PM2.5 (µg/m³)"].isnull()
    & cleaned_df["PM10 (µg/m³)"].isnull()
).sum()

np.int64(33)

In [41]:
pollutants = [
    "PM2.5 (µg/m³)",
    "PM10 (µg/m³)",
    "NO (µg/m³)",
    "NO2 (µg/m³)",
    "SO2 (µg/m³)",
    "CO (mg/m³)",
    "Ozone (µg/m³)"
]

cleaned_df[pollutants].min()

PM2.5 (µg/m³)    1.00
PM10 (µg/m³)     1.00
NO (µg/m³)       0.77
NO2 (µg/m³)      0.80
SO2 (µg/m³)      0.50
CO (mg/m³)       0.03
Ozone (µg/m³)    0.50
dtype: float64

In [43]:
(cleaned_df[pollutants] < 0).sum()

PM2.5 (µg/m³)    0
PM10 (µg/m³)     0
NO (µg/m³)       0
NO2 (µg/m³)      0
SO2 (µg/m³)      0
CO (mg/m³)       0
Ozone (µg/m³)    0
dtype: int64

# 
AT (°C) → Air Temperature
RH (%) → Relative Humidity
WS (m/s) → Wind Speed
WD (deg) → Wind Direction
SR (W/mt2) → Solar Radiation
BP (mmHg) → Barometric Pressure

In [44]:
environmental = [
    "AT (°C)",
    "RH (%)",
    "WS (m/s)",
    "WD (deg)",
    "SR (W/mt2)",
    "BP (mmHg)"
]

cleaned_df[environmental].agg(["min", "max"])

,AT (°C),RH (%),WS (m/s),WD (deg),SR (W/mt2),BP (mmHg)
min,10.76,23.80,0.30,1.0,2.000000,724.540000
max,42.80,93.83,34.57,359.0,1206.134375,975.876042


In [49]:
((cleaned_df["RH (%)"] < 0) |
 (cleaned_df["RH (%)"] > 100)).sum()

((cleaned_df["WD (deg)"] < 0) |
 (cleaned_df["WD (deg)"] > 360)).sum()

(cleaned_df["WS (m/s)"] < 0).sum()

(cleaned_df["SR (W/mt2)"] < 0).sum()

np.int64(0)